<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/02_llms_em_softwares/hands_on_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho Final — LLMs em Sistemas de Software

> **Instruções Gerais**
>
> Este trabalho deve ser entregue como um **Jupyter Notebook (.ipynb)** executado no **Google Colab**.
> Todas as células de código devem estar executadas e com os outputs visíveis no momento da entrega.
>
> - Documente seu raciocínio em células Markdown ao longo do notebook.
> - Use comentários no código para explicar decisões de implementação relevantes.
> - Certifique-se de que o notebook pode ser executado de ponta a ponta sem erros.
> - Variáveis sensíveis (API keys) devem ser carregadas via `google.colab.userdata` ou variáveis de ambiente — **nunca** expostas diretamente no código.

---
---
---

## Tema do Trabalho

Você irá construir um **sistema de perguntas e respostas baseado em RAG (Retrieval-Augmented Generation)** sobre uma base de documentos à sua escolha (exemplos: artigos técnicos, legislação, documentação de software, relatórios, etc.). O sistema deve ser construído com **LangChain**, explorar diferentes estratégias de **prompt engineering** e ser avaliado com métricas objetivas.


---
---
---


## Parte 1 — Prompt Engineering (3 pontos)

### Objetivo
Demonstrar domínio prático de diferentes técnicas de prompting e compreender como os hiperparâmetros do modelo afetam as respostas geradas.

### Tarefas

**1.1 — Comparação de estratégias de prompting**

Escolha uma tarefa de sua preferência (ex: classificação de texto, extração de informação, sumarização, geração de código) e implemente as três estratégias abaixo para a **mesma tarefa e o mesmo modelo**:

- **Zero-shot**: apenas a instrução, sem exemplos.
- **Few-shot**: instrução com 3 exemplos no prompt.
- **Chain-of-Thought (CoT)**: instrução que induz o modelo a raciocinar passo a passo antes de responder.

Para cada estratégia, execute ao menos **3 inputs diferentes** e registre as respostas.

**1.2 — Análise dos hiperparâmetros**

Usando a mesma tarefa do item 1.1, varie os seguintes parâmetros e registre o impacto observado nas respostas:

| Parâmetro     | Valores a testar         |
|---------------|--------------------------|
| `temperature` | 0.0 / 0.7 / 1.4          |
| `top_p`       | 0.5 / 0.9 / 1.0          |
| `max_tokens`  | Restritivo / Adequado / Amplo |

**1.3 — Discussão**

Em uma célula Markdown, responda:
- Qual estratégia apresentou os melhores resultados para a sua tarefa? Por quê?
- Como a variação de `temperature` impactou a consistência e criatividade das respostas?
- Quais limitações você identificou em cada abordagem?

### Critérios de Avaliação
- Implementação correta das três estratégias (1 pt)
- Experimento com hiperparâmetros documentado e com outputs visíveis (1 pt)
- Qualidade e profundidade da análise crítica (1 pt)


In [ ]:
# adicione seu código e suas respostas aqui

In [ ]:
# adicione seu código e suas respostas aqui

In [ ]:
# adicione seu código e suas respostas aqui

## Parte 2 — Construção do Pipeline RAG com LangChain (3,5 pontos)

### Objetivo
Construir um pipeline RAG completo e funcional utilizando LangChain, integrando carregamento de documentos, chunking, embeddings, banco vetorial e recuperação.

### Tarefas

**2.1 — Preparação da base de dados**

- Escolha uma coleção de documentos (mínimo de **3 arquivos** em PDF, TXT ou CSV).
- Carregue os documentos utilizando os **Document Loaders** do LangChain.
- Aplique **uma estratégia de chunking** (ex: `RecursiveCharacterTextSplitter` com tamanhos distintos, ou `CharacterTextSplitter` ou chunking por parágrafo) e justifique a escolha final.

**2.2 — Indexação vetorial**

- Gere embeddings com um modelo à sua escolha (ex: OpenAI `text-embedding-ada-002`, HuggingFace `sentence-transformers`, etc.).
- Indexe os chunks em um banco vetorial (FAISS, Chroma ou Pinecone).
- Demonstre uma busca de similaridade direta no banco vetorial com ao menos **2 queries de teste**, exibindo os chunks recuperados.

**2.3 — Pipeline de geração**

- Monte uma chain de RAG com LangChain (`RetrievalQA` ou `ConversationalRetrievalChain`) que:
  - Recupere os `top-k` chunks mais relevantes (experimente ao menos dois valores de `k`).
  - Injete o contexto recuperado em um prompt estruturado.
  - Gere a resposta final com o LLM de sua escolha.
- Execute ao menos **3 perguntas** sobre a sua base de documentos e exiba as respostas com os respectivos trechos de contexto utilizados.

### Critérios de Avaliação
- Carregamento, chunking e indexação corretos e justificados (1,5 pt)
- Pipeline RAG funcional com contexto injetado corretamente (1,5 pt)
- Diversidade de queries testadas e clareza nos outputs (0,5 pt)


## Parte 2 - Construção do Pipeline RAG com LangChain

## Tarefa 2.1: Preparação da base de dados

Instalamos todas as bibliotecas necessárias para o pipeline RAG.

In [ ]:
!pip install -q groq==0.13.0 langchain-community langchain-core>=1.4.0 langchain-text-splitters faiss-cpu pypdf sentence-transformers rouge-score==0.1.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### 2.1.1 Configuração da API Key da Groq

Carregamos a chave da API Groq de forma segura.

In [ ]:
import os, json, time
from groq import Groq

# Carrega a chave de forma segura a partir dos Secrets do Colab
# Se não estiver no Colab, usa variável de ambiente como fallback
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "SUA_CHAVE_AQUI")

# Configuramos a chave como variável de ambiente (necessária para langchain-groq)
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Inicializamos o cliente Groq (útil para chamadas diretas, ex: Parte 3)
client = Groq(api_key=GROQ_API_KEY)

# Definimos os modelos que serão utilizados
MODEL_RAG = "llama-3.1-8b-instant"      # Rápido, para gerar respostas no RAG

print("✅ Configuração Groq OK!")
print(f"   Model : {MODEL_RAG}")

✅ Configuração Groq OK!
   Model : llama-3.1-8b-instant


###  2.1.2 Preparação da base de dados

Descrição:

Nesta etapa realizamos três operações fundamentais:



* CRIAÇÃO DOS DOCUMENTOS DE EXEMPLO: Como o enunciado pede no mínimo 3 arquivos, criamos 3 documentos .txt sobre o tema "Inteligência Artificial" como exemplo. Na prática, você pode substituir por seus próprios PDFs, TXTs ou CSVs.
* CARREGAMENTO COM DOCUMENT LOADERS: Usamos DirectoryLoader + TextLoader do LangChain para carregar todos os arquivos .txt de uma pasta de forma automática.
* CHUNKING (DIVISÃO EM TRECHOS): Usamos RecursiveCharacterTextSplitter, que é a estratégia que divide primeiro por parágrafos (\n\n), depois por frases (\n), e por último por caracteres individuais.

### 2.1.3 Criação dos documentos de exemplo

In [ ]:
# Criamos uma pasta "documentos" e salvamos 3 arquivos de texto sobre IA.

os.makedirs("documentos", exist_ok=True)

# Documento 1: História da IA
doc1 = """Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da computação que busca
criar sistemas capazes de realizar tarefas que normalmente requerem inteligência
humana. O termo foi cunhado por John McCarthy em 1956, durante a conferência de
Dartmouth, considerada o marco fundador da área.

Nas décadas de 1960 e 1970, os pesquisadores desenvolveram sistemas especialistas,
que eram programas baseados em regras lógicas para resolver problemas específicos.
No entanto, essas abordagens tinham limitações significativas na generalização.

O chamado "inverno da IA" ocorreu nos anos 1980 e início dos anos 1990, quando o
financiamento e o interesse na área diminuíram drasticamente devido às expectativas
não cumpridas. A recuperação veio com o avanço do poder computacional e o
surgimento de técnicas de aprendizado de máquina.

A partir de 2010, o deep learning revolucionou a IA, permitindo avanços
significativos em reconhecimento de imagens, processamento de linguagem natural
e jogos. Marcos importantes incluem a vitória do AlphaGo sobre o campeão mundial
de Go em 2016 e o lançamento do GPT-3 em 2020.
"""

# Documento 2: Tipos de IA
doc2 = """Tipos de Inteligência Artificial

A IA pode ser classificada em diferentes categorias com base em suas capacidades:

1. IA Estreita (Narrow AI ou ANI):
É o tipo de IA que existe atualmente. São sistemas projetados para realizar
tarefas específicas, como reconhecimento facial, tradução automática ou
recomendação de conteúdo. Exemplos incluem Siri, Alexa e sistemas de GPS.
A ANI não possui consciência nem compreensão real do que faz.

2. IA Geral (AGI - Artificial General Intelligence):
É uma IA hipotética que teria capacidades cognitivas equivalentes às humanas,
podendo aprender e realizar qualquer tarefa intelectual que um ser humano pode.
Ainda não foi alcançada e é tema de intenso debate na comunidade científica.

3. Superinteligência Artificial (ASI):
É um conceito teórico de uma IA que superaria a inteligência humana em todos
os aspectos, incluindo criatividade, resolução de problemas e habilidades
sociais. Pesquisadores como Nick Bostrom alertam para os riscos potenciais.

Machine Learning (Aprendizado de Máquina):
É um subcampo da IA que permite que sistemas aprendam a partir de dados sem
serem explicitamente programados. Os três paradigmas principais são:
- Aprendizado Supervisionado: treina com dados rotulados
- Aprendizado Não Supervisionado: encontra padrões em dados sem rótulos
- Aprendizado por Reforço: aprende através de tentativa e erro com recompensas
"""

# Documento 3: Aplicações e ética
doc3 = """Aplicações e Ética da Inteligência Artificial

Aplicações Práticas da IA:

Saúde: A IA é utilizada para diagnóstico médico assistido, análise de imagens
de exames (raio-X, ressonância magnética), descoberta de novos medicamentos
e monitoramento remoto de pacientes. Modelos de deep learning conseguem
detectar certos tipos de câncer com precisão comparável à de médicos especialistas.

Educação: Sistemas de tutoria inteligente adaptam o conteúdo ao ritmo de
aprendizado de cada aluno. Plataformas como Duolingo usam IA para personalizar
exercícios e otimizar a retenção de conhecimento.

Transporte: Veículos autônomos utilizam múltiplos sensores e algoritmos de IA
para navegar. Empresas como Tesla, Waymo e Cruise investem bilhões nessa tecnologia.
A IA também otimiza rotas logísticas e gestão de tráfego urbano.

Finanças: Algoritmos de IA detectam fraudes em transações bancárias, fazem
previsões de mercado e automatizam processos de análise de crédito.

Questões Éticas:

Viés Algorítmico: Modelos de IA podem perpetuar e amplificar preconceitos
presentes nos dados de treinamento, levando a decisões discriminatórias em
áreas como contratação, crédito e justiça criminal.

Privacidade: O uso massivo de dados pessoais para treinar modelos de IA
levanta preocupações sobre vigilância e uso indevido de informações.

Impacto no Emprego: A automação impulsionada pela IA pode substituir empregos
em diversos setores, exigindo políticas de requalificação profissional.

Regulamentação: A União Europeia propôs o AI Act, uma das primeiras legislações
abrangentes para regular o desenvolvimento e uso de sistemas de IA, classificando
aplicações por nível de risco.
"""

# Salvamos cada documento em um arquivo separado
with open("documentos/historia_ia.txt", "w", encoding="utf-8") as f:
    f.write(doc1)

with open("documentos/tipos_ia.txt", "w", encoding="utf-8") as f:
    f.write(doc2)

with open("documentos/aplicacoes_etica_ia.txt", "w", encoding="utf-8") as f:
    f.write(doc3)

print("3 documentos criados na pasta 'documentos/':")
for arquivo in os.listdir("documentos"):
    print(f"    {arquivo}")

3 documentos criados na pasta 'documentos/':
    historia_ia.txt
    tipos_ia.txt
    aplicacoes_etica_ia.txt


### 2.1.4 Carregamento dos documentos com Document Loaders

O DirectoryLoader percorre uma pasta e carrega todos os arquivos que correspondem ao padrão glob (neste caso, *.txt). Cada arquivo é convertido em um objeto Document do LangChain, que contém o texto (page_content) e metadados (metadata).

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Configuramos o loader para ler todos os arquivos .txt da pasta "documentos"
# O parâmetro loader_kwargs define a codificação UTF-8 para suportar acentos
loader = DirectoryLoader(
    path="documentos",            # Pasta onde estão os documentos
    glob="**/*.txt",              # Padrão: todos os .txt, incluindo subpastas
    loader_cls=TextLoader,        # Classe de carregamento para arquivos de texto
    loader_kwargs={"encoding": "utf-8"}  # Codificação para caracteres especiais
)

# Executamos o carregamento
documentos = loader.load()

print(f"  Total de documentos carregados: {len(documentos)}")
print()

# Exibimos informações sobre cada documento carregado
for i, doc in enumerate(documentos):
    print(f"   Documento {i+1}:")
    print(f"   Arquivo: {doc.metadata['source']}")
    print(f"   Tamanho: {len(doc.page_content)} caracteres")
    print(f"   Prévia:  {doc.page_content[:100]}...")
    print()

  Total de documentos carregados: 3

   Documento 1:
   Arquivo: documentos/historia_ia.txt
   Tamanho: 1150 caracteres
   Prévia:  Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da...

   Documento 2:
   Arquivo: documentos/tipos_ia.txt
   Tamanho: 1389 caracteres
   Prévia:  Tipos de Inteligência Artificial

A IA pode ser classificada em diferentes categorias com base em su...

   Documento 3:
   Arquivo: documentos/aplicacoes_etica_ia.txt
   Tamanho: 1668 caracteres
   Prévia:  Aplicações e Ética da Inteligência Artificial

Aplicações Práticas da IA:

Saúde: A IA é utilizada p...



### 2.1.5 Chunking (Divisão em trechos menores)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuramos o text splitter com os parâmetros escolhidos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Tamanho máximo de cada chunk (em caracteres)
    chunk_overlap=50,     # Sobreposição entre chunks consecutivos
    length_function=len,  # Função para medir o tamanho (contagem de caracteres)
    separators=[          # Hierarquia de separadores (do mais preferido ao menos)
        "\n\n",           # Primeiro tenta: separar por parágrafos
        "\n",             # Depois: por quebras de linha
        " ",              # Depois: por espaços (entre palavras)
        ""                # Último recurso: caractere por caractere
    ]
)

In [ ]:
# Aplicamos o chunking em todos os documentos carregados
chunks = text_splitter.split_documents(documentos)

print(f"Chunking concluído!")
print(f"   Documentos originais: {len(documentos)}")
print(f"   Chunks gerados: {len(chunks)}")
print(f"   Tamanho configurado: 500 caracteres (com 50 de overlap)")
print()

# Exibimos os primeiros 5 chunks para verificação
print("=" * 70)
print("PRÉVIA DOS PRIMEIROS 5 CHUNKS:")
print("=" * 70)
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1} (fonte: {chunk.metadata['source']}):")
    print(f"   Tamanho: {len(chunk.page_content)} caracteres")
    print(f"   Conteúdo:\n   {chunk.page_content[:200]}...")
    print("-" * 70)

Chunking concluído!
   Documentos originais: 3
   Chunks gerados: 12
   Tamanho configurado: 500 caracteres (com 50 de overlap)

PRÉVIA DOS PRIMEIROS 5 CHUNKS:

Chunk 1 (fonte: documentos/historia_ia.txt):
   Tamanho: 331 caracteres
   Conteúdo:
   Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da computação que busca
criar sistemas capazes de realizar tarefas que normalmente requerem inteligênc...
----------------------------------------------------------------------

Chunk 2 (fonte: documentos/historia_ia.txt):
   Tamanho: 245 caracteres
   Conteúdo:
   Nas décadas de 1960 e 1970, os pesquisadores desenvolveram sistemas especialistas,
que eram programas baseados em regras lógicas para resolver problemas específicos.
No entanto, essas abordagens tinha...
----------------------------------------------------------------------

Chunk 3 (fonte: documentos/historia_ia.txt):
   Tamanho: 288 caracteres
   Conteúdo:
   O chamado "inverno da I

## Tarefa 2.2: Indexação Vetorial

Descrição:
Nesta etapa transformamos os chunks de texto em vetores numéricos (embeddings) e os armazenamos em um banco vetorial (FAISS) para permitir buscas por similaridade semântica.

### 2.2.1 Geração de Embeddings e criação do banco vetorial FAISS

O FAISS é criado diretamente a partir dos chunks, gerando os embeddings
automaticamente e indexando tudo de uma vez.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
# Inicializamos o modelo de embeddings multilíngue do HuggingFace
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},      # Usa CPU
    encode_kwargs={"normalize_embeddings": True}  # Normaliza para similaridade de cosseno
)

print("Carregando modelo de embeddings multilíngue...")

banco_vetorial = FAISS.from_documents(
    documents=chunks,            # Lista de chunks a indexar
    embedding=modelo_embeddings  # Modelo para gerar os embeddings
)

print(f"   Banco vetorial FAISS criado com sucesso!")
print(f"   Total de vetores indexados: {len(chunks)}")
print(f"   Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando modelo de embeddings multilíngue...
   Banco vetorial FAISS criado com sucesso!
   Total de vetores indexados: 12
   Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2


### 2.2.2 Demonstração de busca por similaridade

Realizamos duas buscas de teste para verificar que o banco vetorial
está funcionando corretamente e retornando chunks relevantes.

O método similarity_search_with_score retorna:
- Os k chunks mais similares à query
- O score de distância (quanto MENOR, mais similar — no FAISS usa L2)

In [ ]:
query_1 = "Quando surgiu o termo Inteligência Artificial?"

print("=" * 70)
print(f"🔍 QUERY 1: \"{query_1}\"")
print("=" * 70)

# Buscamos os 3 chunks mais similares
resultados_1 = banco_vetorial.similarity_search_with_score(query_1, k=3)

for i, (doc, score) in enumerate(resultados_1):
    print(f"\n📌 Resultado {i+1} (distância L2: {score:.4f}):")
    print(f"   Fonte: {doc.metadata['source']}")
    print(f"   Conteúdo: {doc.page_content[:250]}...")
    print("-" * 70)

# Query de teste 2: sobre ética na IA
query_2 = "Quais são os problemas éticos da inteligência artificial?"

print()
print("=" * 70)
print(f"🔍 QUERY 2: \"{query_2}\"")
print("=" * 70)

# Buscamos os 3 chunks mais similares
resultados_2 = banco_vetorial.similarity_search_with_score(query_2, k=3)

for i, (doc, score) in enumerate(resultados_2):
    print(f"\n📌 Resultado {i+1} (distância L2: {score:.4f}):")
    print(f"   Fonte: {doc.metadata['source']}")
    print(f"   Conteúdo: {doc.page_content[:250]}...")
    print("-" * 70)

🔍 QUERY 1: "Quando surgiu o termo Inteligência Artificial?"

📌 Resultado 1 (distância L2: 0.3410):
   Fonte: documentos/historia_ia.txt
   Conteúdo: Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da computação que busca
criar sistemas capazes de realizar tarefas que normalmente requerem inteligência
humana. O termo foi cunhado por John McCarthy e...
----------------------------------------------------------------------

📌 Resultado 2 (distância L2: 0.6569):
   Fonte: documentos/historia_ia.txt
   Conteúdo: O chamado "inverno da IA" ocorreu nos anos 1980 e início dos anos 1990, quando o
financiamento e o interesse na área diminuíram drasticamente devido às expectativas
não cumpridas. A recuperação veio com o avanço do poder computacional e o
surgimento ...
----------------------------------------------------------------------

📌 Resultado 3 (distância L2: 0.7205):
   Fonte: documentos/tipos_ia.txt
   Conteúdo: 2. IA Geral (AGI - Artific

## Parte 3 — Avaliação do Sistema (3,5 pontos)

### Objetivo
Avaliar a qualidade do sistema RAG construído utilizando métricas quantitativas e qualitativas.

### Tarefas

**3.1 — Conjunto de avaliação**

- Crie um conjunto de **10 pares (pergunta, resposta esperada)** manualmente anotados, com base nos documentos da sua base. Organize-os em um dicionário ou DataFrame.

**3.2 — Métricas automáticas**

Para cada par do conjunto de avaliação, calcule:

- **ROUGE-L** entre a resposta gerada e a resposta esperada.
- **Similaridade semântica** (cosseno) entre os embeddings da resposta gerada e da resposta esperada.

Apresente os resultados em uma tabela e calcule a média geral.

**3.3 — LLM-as-a-judge**

- Implemente um avaliador baseado em LLM que, para cada resposta gerada, atribua uma nota de **1 a 5** (escala Likert) nos seguintes critérios:
  - **Fidelidade ao contexto** (*groundedness*): a resposta está ancorada nos documentos recuperados?
  - **Relevância**: a resposta responde de fato à pergunta?
  - **Completude**: a resposta abrange os pontos principais?
- Exiba a distribuição das notas em um gráfico.

### Critérios de Avaliação
- Conjunto de avaliação bem definido e métricas automáticas calculadas corretamente (1,75 pt)
- LLM-as-a-judge implementado com critérios claros e distribuição visualizada (1,75 pt)


In [ ]:
# adicione seu código e suas respostas aqui

In [ ]:
# adicione seu código e suas respostas aqui

In [ ]:
# adicione seu código e suas respostas aqui

## Entrega

| Item | Detalhe |
|------|---------|
| **Formato** | Arquivo `.ipynb` com todas as células executadas |
| **Nome do arquivo** | `trabalho_final_[seu_nome].ipynb` |
| **Prazo** | Conforme comunicado pelo professor |
| **Plataforma** | Submissão via google classroom |

> Notebooks com células não executadas, sem outputs ou que gerem erros ao rodar serão penalizados.
> A presença de API keys expostas diretamente no código implica desconto de **1 ponto**.
